# CORAL Step-by-Step Tutorial: Saccharomyces Example

This tutorial runs each component of the CORAL trio pipeline separately, on two Saccharomyces sisters and an outgroup.
Outputs are written to `../test_output/step_by_step_tutorial/`. For production use, run the whole pipeline with
`coral run_trio` or `MutationExtractionPipeline`.

In [ ]:
import os
from coral.genome_manager import Genome
from coral.alignment_manager import ALIGNERS, Aligner
from coral.bam_extractor import BamTrioExtractor
from coral.mutation_extractor_manager import MutationNormalizer
from coral.pipeline import MutationExtractionPipeline
from coral.plot_utils import MutationSpectraPlotter, CoveragePlotter, MutationDensityPlotter
from coral.utils import get_top_n_chromosomes

BASE_OUTPUT_DIR = "../test_output/step_by_step_tutorial"
OUTGROUP = ("Saccharomyces_mikatae_IFO_1815", "GCF_947241705.1")
SPECIES1 = ("Saccharomyces_paradoxus", "GCF_002079055.1")
SPECIES2 = ("Saccharomyces_cerevisiae_S288C", "GCF_000146045.2")
MAPQ = 60
CORES = None  # all available cores
VERBOSE = True
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

## Step 1: Download genomes

Genomes are downloaded from NCBI with the datasets CLI, one FASTA per species.

In [ ]:
reference, genome1, genome2 = (Genome(name=name, accession=accession, output_dir=BASE_OUTPUT_DIR, verbose=VERBOSE)
                               for name, accession in (OUTGROUP, SPECIES1, SPECIES2))
for genome in (reference, genome1, genome2):
    genome.download()

## Step 2: Index the reference genome

The outgroup is the reference. It is indexed once, for BWA and for samtools.

In [ ]:
reference.index(ALIGNERS["bwa"]["index"])

## Step 3: Fragment the genomes into pseudo-reads

Each sister genome is fragmented into 150 bp pseudo-reads with a 75 bp step (~2x tiling coverage), written as FASTQ.

In [ ]:
for genome in (genome1, genome2):
    genome.generate_fragment_fastq(length=150, offset=75)

## Step 4: Align the pseudo-reads to the reference

The pseudo-reads are aligned with BWA-MEM. Pseudo-reads with MAPQ 60 are kept, and pseudo-reads with 1 ≤ MAPQ < 60
only if they overlap a neighbouring pseudo-read of the same genome on the reference (the continuity rule).

In [ ]:
aligners = [Aligner(species_genome=genome, reference_genome=reference, base_output_dir=BASE_OUTPUT_DIR,
                    aligner_name="bwa", cores=CORES, verbose=VERBOSE) for genome in (genome1, genome2)]
for aligner in aligners:
    aligner.align_disk_cached(mapq=MAPQ, low_mapq=1, continuity=True)

## Step 5: Extract mutations

Both sisters' alignments are parsed along the outgroup genome. A site is callable when it is covered by at least one
retained pseudo-read, all reads agree on the base and none carries an insertion or deletion there. A mutation is
called where the flanking bases are identical in all three species, and is assigned to the sister whose central base
differs from the shared base of the other sister and the outgroup. `five_mer=True` counts the same calls by 5-mer
context (`Five_mers/`), in a second pass.

In [ ]:
run_id = f"{OUTGROUP[0]}__{SPECIES1[0]}__{SPECIES2[0]}"
run_dir = os.path.join(BASE_OUTPUT_DIR, run_id)
mutations_dir, triplets_dir = os.path.join(run_dir, "Mutations"), os.path.join(run_dir, "Triplets")
for five_mer in (False, True):
    BamTrioExtractor(reference=reference.name, taxon1=genome1.name, taxon2=genome2.name,
                     ref_fasta=reference.fasta_path, bams=[a.final_bam for a in aligners],
                     mutation_output_dir=mutations_dir, triplet_output_dir=triplets_dir,
                     cores=CORES, five_mer=five_mer, verbose=VERBOSE).extract()

## Step 6: Normalize mutation counts

Counts are normalized by the trinucleotide abundance at callable sites (counted on the outgroup), and the 192
substitution types are collapsed into the standard 96 categories (`Tables/`).

In [ ]:
tables_dir = os.path.join(run_dir, "Tables")
MutationNormalizer(input_dir=run_dir, output_dir=tables_dir, verbose=VERBOSE).normalize()

## Step 7: Plots

Mutation spectra, and coverage and mutation density along the three largest chromosomes. The coverage plots use the
read intervals extracted from the BAMs (`Intervals/`).

In [ ]:
plots_dir = os.path.join(run_dir, "Plots")
os.makedirs(plots_dir, exist_ok=True)
MutationSpectraPlotter().plot(tables_dir=tables_dir, output_dir=plots_dir, verbose=VERBOSE)

pipeline = MutationExtractionPipeline(species_list=[SPECIES1, SPECIES2], outgroup=OUTGROUP,
                                      base_output_dir=BASE_OUTPUT_DIR, run_id=run_id, verbose=VERBOSE)
pipeline.alignments = aligners
pipeline.extract_intervals()

fai_file = reference.fasta_path + ".fai"
coverage_plotter = CoveragePlotter(fai_file=fai_file, verbose=VERBOSE)
density_plotter = MutationDensityPlotter(fai_file=fai_file, verbose=VERBOSE)
for chrom in get_top_n_chromosomes(fai_file, n=3):
    coverage_plotter.plot(interval_dir=os.path.join(run_dir, "Intervals"), chromosome=chrom,
                          output_dir=os.path.join(plots_dir, f"coverage_{chrom}.png"))
    density_plotter.plot(mutation_dir=mutations_dir, chromosome=chrom, output_dir=plots_dir)

## Outputs

```
step_by_step_tutorial/
  ├── <species>/                 genome FASTA, index and pseudo-read FASTQ
  ├── BAMs/                      filtered alignments
  └── <run_id>/
      ├── Mutations/             mutation lists (CSV) and counts (JSON)
      ├── Triplets/              trinucleotide counts at callable sites
      ├── Five_mers/             5-mer counts
      ├── Tables/                normalized spectra
      ├── Intervals/             read intervals
      ├── Plots/                 plots
      └── run_summary.json       extraction diagnostics
```

See [OUTPUT_FORMAT.md](OUTPUT_FORMAT.md) for the file formats.